# ⚡ EPİAŞ PTF Forecast - Dedicated Model Experiments Notebook

**Proje:** Türkiye Elektrik Piyasası Piyasa Takas Fiyatı (PTF - USD/MWh & TL/MWh) Tahmini  
**Görev (Task):** Günlük Gün Öncesi Piyasası (GÖP) için yarının **24 saatlik fiyat bloğunun** tahmin edilmesi.  
**Metrikler:** **MAE ($/MWh)**, **MAPE (%)** ve **WAPE (% - Weighted Absolute Percentage Error)**  
**Doğrulama Stratejisi (Backtest):** Her gün geriye giderek modeli son 6 ayın verisiyle eğitip yarının 24 saatini tahmin eden **Günlük Walk-Forward Backtest**.  
**Raporlama Ufku:** Geriye dönük **Son 3 Ay (90 Gün)**, **Son 6 Ay (180 Gün)**, **Son 9 Ay (270 Gün)** ve **Son 12 Ay (365 Gün)** Metrik Ortalamaları.

---


## 1. Kütüphanelerin Yüklenmesi ve Veritabanı Bağlantısı


In [1]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sqlalchemy import text
from sklearn.metrics import mean_squared_error, mean_absolute_error
import lightgbm as lgb

project_root = Path.cwd().parent if Path.cwd().name == 'eda' else Path.cwd()
sys.path.insert(0, str(project_root))

from db.connection import get_db_engine

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

print("✅ Kütüphaneler ve veri bağlantı modülleri başarıyla yüklendi.")

✅ Kütüphaneler ve veri bağlantı modülleri başarıyla yüklendi.


## 2. Silver Katmanı Verilerinin Birleştirilmesi (Baraj Su İmkânı Dahil Master Dataset)


In [2]:
engine = get_db_engine()

master_sql = text("""
    SELECT 
        m.ts,
        m.price_usd AS mcp_price_usd,
        m.price_try AS mcp_price_try,
        s.system_marginal_price_try AS smp_price_try,
        l.load_forecast_mw,
        k.total_mw AS kgup_total_mw,
        k.natural_gas_mw AS kgup_gas_mw,
        k.wind_mw AS kgup_wind_mw,
        k.solar_mw AS kgup_solar_mw,
        k.dammed_hydro_mw + k.river_hydro_mw AS kgup_hydro_mw,
        k.import_coal_mw + k.lignite_mw + k.black_coal_mw AS kgup_coal_mw,
        g.total_mw AS actual_gen_total_mw,
        c.consumption_mw AS actual_cons_mw,
        w.turkey_weighted_temperature_c AS temperature_c,
        mc.usd_try,
        mc.brent_oil_usd,
        ng.gas_reference_price_try AS natural_gas_grf_try,
        wp.hydro_water_energy_mwh
    FROM raw_mcp_hourly m
    LEFT JOIN raw_smp_hourly s ON m.ts = s.ts
    LEFT JOIN raw_load_forecast_hourly l ON m.ts = l.ts
    LEFT JOIN raw_kgup_hourly k ON m.ts = k.ts
    LEFT JOIN raw_actual_generation_hourly g ON m.ts = g.ts
    LEFT JOIN raw_actual_consumption_hourly c ON m.ts = c.ts
    LEFT JOIN raw_weather_hourly w ON m.ts = w.ts
    LEFT JOIN raw_macro_daily mc ON DATE(m.ts) = mc.entry_date
    LEFT JOIN raw_natural_gas_daily ng ON DATE(m.ts) = ng.entry_date
    LEFT JOIN (
        SELECT DATE(date_time) AS entry_date, SUM(water_energy_provision_mwh) AS hydro_water_energy_mwh
        FROM raw_master_water_energy_provision
        GROUP BY DATE(date_time)
    ) wp ON DATE(m.ts) = wp.entry_date
    ORDER BY m.ts ASC;
""")

with engine.connect() as conn:
    df_raw = pd.read_sql(master_sql, conn)

df_raw['ts'] = pd.to_datetime(df_raw['ts']).dt.tz_convert('Europe/Istanbul')
df_raw = df_raw.set_index('ts').sort_index()

df_raw['usd_try'] = df_raw['usd_try'].ffill().bfill()
df_raw['brent_oil_usd'] = df_raw['brent_oil_usd'].ffill().bfill()
df_raw['natural_gas_grf_try'] = df_raw['natural_gas_grf_try'].ffill().bfill()
if 'hydro_water_energy_mwh' in df_raw.columns:
    df_raw['hydro_water_energy_mwh'] = df_raw['hydro_water_energy_mwh'].ffill().bfill()

print(f"📊 Yüklenen Toplam Saatlik Satır Sayısı: {len(df_raw):,} saat ({df_raw.index.min()} ile {df_raw.index.max()} arası)")

📊 Yüklenen Toplam Saatlik Satır Sayısı: 22,584 saat (2024-01-01 00:00:00+03:00 ile 2026-07-29 23:00:00+03:00 arası)


## 3. Gelecek Sızıntısız Öznitelik Mühendisliği (Data Leakage Free Pipeline)


In [3]:
def build_features(df):
    df_feat = df.copy()
    
    # A. Takvim Öznitelikleri
    df_feat['hour'] = df_feat.index.hour
    df_feat['dayofweek'] = df_feat.index.dayofweek
    df_feat['month'] = df_feat.index.month
    df_feat['quarter'] = df_feat.index.quarter
    df_feat['is_weekend'] = (df_feat.index.dayofweek >= 5).astype(int)
    df_feat['is_peak_hour'] = df_feat['hour'].isin([17, 18, 19, 20, 21]).astype(int)
    
    # B. Gecikmeli Fiyat Öznitelikleri (PTF en az 24 saat gecikme: T-1 biliniyor)
    for lag in [24, 48, 168]:
        df_feat[f'mcp_usd_lag_{lag}'] = df_feat['mcp_price_usd'].shift(lag)
    
    # C. Gecikmeli Plan ve Tahmin Öznitelikleri (En az 24 saat / 1 gün gecikmeli - T-1 planı)
    for lag in [24, 48, 168]:
        df_feat[f'load_lag_{lag}'] = df_feat['load_forecast_mw'].shift(lag)
        df_feat[f'kgup_lag_{lag}'] = df_feat['kgup_total_mw'].shift(lag)
        df_feat[f'kgup_wind_lag_{lag}'] = df_feat['kgup_wind_mw'].shift(lag)
        df_feat[f'kgup_solar_lag_{lag}'] = df_feat['kgup_solar_mw'].shift(lag)
        df_feat[f'kgup_hydro_lag_{lag}'] = df_feat['kgup_hydro_mw'].shift(lag)
        df_feat[f'kgup_gas_lag_{lag}'] = df_feat['kgup_gas_mw'].shift(lag)
    
    # D. Gecikmeli Gerçekleşen Veriler (Gerçekleşen Tüketim/Üretim en az 48 saat gecikme: T-2 biliniyor)
    for lag in [48, 168]:
        if 'actual_gen_total_mw' in df_feat.columns:
            df_feat[f'actual_gen_lag_{lag}'] = df_feat['actual_gen_total_mw'].shift(lag)
        if 'actual_cons_mw' in df_feat.columns:
            df_feat[f'actual_cons_lag_{lag}'] = df_feat['actual_cons_mw'].shift(lag)
    
    # E. Hareketli İstatistikler
    df_feat['mcp_usd_roll_mean_24h'] = df_feat['mcp_price_usd'].shift(24).rolling(window=24).mean()
    df_feat['mcp_usd_roll_std_24h'] = df_feat['mcp_price_usd'].shift(24).rolling(window=24).std()
    df_feat['mcp_usd_roll_mean_7d'] = df_feat['mcp_price_usd'].shift(24).rolling(window=168).mean()
    
    # F. Arz-Talep Dengesi & Yenilenebilir Oranlar (1 Gün Önceki Plan/KGÜP Verisiyle - shift(24))
    df_feat['supply_demand_gap_mw'] = df_feat['load_lag_24'] - df_feat['kgup_lag_24']
    total_kgup_safe = df_feat['kgup_lag_24'].replace(0, np.nan)
    df_feat['renewable_ratio'] = (df_feat['kgup_wind_lag_24'] + df_feat['kgup_solar_lag_24'] + df_feat['kgup_hydro_lag_24']) / total_kgup_safe
    df_feat['renewable_ratio'] = df_feat['renewable_ratio'].fillna(0)
    
    # G. Güneş & Rüzgar Zirve Oranı
    df_feat['solar_wind_ratio'] = (df_feat['kgup_wind_lag_24'] + df_feat['kgup_solar_lag_24']) / total_kgup_safe
    df_feat['solar_wind_ratio'] = df_feat['solar_wind_ratio'].fillna(0)
    
    return df_feat

df_feat = build_features(df_raw)
df_model = df_feat.dropna().copy()

target_col = 'mcp_price_usd'
exclude_cols = [
    'mcp_price_usd', 'mcp_price_try', 'smp_price_try', 
    'actual_gen_total_mw', 'actual_cons_mw',
    'load_forecast_mw', 'kgup_total_mw', 'kgup_gas_mw', 
    'kgup_wind_mw', 'kgup_solar_mw', 'kgup_hydro_mw', 'kgup_coal_mw'
]
feature_cols = [c for c in df_model.columns if c not in exclude_cols]

# Metrik Hesaplama Fonksiyonları
def calculate_safe_mape(y_true, y_pred):
    safe_denom = np.maximum(y_true, 1.0)
    return np.mean(np.abs((y_true - y_pred) / safe_denom)) * 100

def calculate_wape(y_true, y_pred):
    sum_actual = np.sum(np.abs(y_true))
    if sum_actual == 0:
        return 0.0
    return (np.sum(np.abs(y_true - y_pred)) / sum_actual) * 100

print(f"✨ Temizlenmiş Model Dataseti Hazır: {len(df_model):,} saat | Değişken Sayısı: {len(feature_cols)}")

✨ Temizlenmiş Model Dataseti Hazır: 22,277 saat | Değişken Sayısı: 42


## 4. Genel Günlük Walk-Forward Backtest Motoru (MAE, MAPE & WAPE Metrikleriyle)

Tüm model denemelerinde standart olarak kullanılacak **3, 6, 9 ve 12 Aylık (365 Gün) Günlük Backtest** motoru.


In [4]:
def run_daily_walk_forward_backtest(df_data, feature_list, target_name, train_predict_fn, max_days=365, lookback_hours=4380):
    daily_results = []
    
    for day_idx in range(max_days):
        test_end = df_data.index.max() - pd.Timedelta(days=day_idx)
        test_start = test_end - pd.Timedelta(hours=23)
        train_end = test_start - pd.Timedelta(hours=1)
        train_start = train_end - pd.Timedelta(hours=lookback_hours)
        
        tr_df = df_data.loc[train_start:train_end]
        te_df = df_data.loc[test_start:test_end]
        
        if len(tr_df) < 1000 or len(te_df) < 12:
            continue
            
        preds = train_predict_fn(tr_df, te_df, feature_list, target_name)
        
        y_true = te_df[target_name].values
        mae = mean_absolute_error(y_true, preds)
        mape = calculate_safe_mape(y_true, preds)
        wape = calculate_wape(y_true, preds)
        
        daily_results.append({
            'day_offset': day_idx,
            'tarih': test_start.strftime('%Y-%m-%d'),
            'mae': mae,
            'mape': mape,
            'wape': wape
        })
        
        if (day_idx + 1) % 5 == 0 or (day_idx + 1) == max_days or max_days <= 10:
            pct = ((day_idx + 1) / max_days) * 100
            print(f"⏳ [{day_idx+1:02d}/{max_days:02d}] ({pct:5.1f}%) | Tarih: {test_start.strftime('%Y-%m-%d')} | MAE: ${mae:5.2f}/MWh | WAPE: %{wape:5.2f}")
        
    res_df = pd.DataFrame(daily_results)
    
    horizon_summary = []
    for m_num, d_cnt in [(3, 90), (6, 180), (9, 270), (12, 365)]:
        sub = res_df.iloc[:min(d_cnt, len(res_df))]
        horizon_summary.append({
            'Test Dönemi': f'Son {m_num} Ay ({len(sub)} Gün)',
            'Ortalama MAE ($/MWh)': round(sub['mae'].mean(), 2),
            'Ortalama MAPE (%)': round(sub['mape'].mean(), 2),
            'Ortalama WAPE (%)': round(sub['wape'].mean(), 2)
        })
        
    return pd.DataFrame(horizon_summary), res_df

print("⚙️ WAPE Destekli Günlük Walk-Forward Backtest Motoru Yüklendi.")

⚙️ WAPE Destekli Günlük Walk-Forward Backtest Motoru Yüklendi.


## 🧪 DENEME 1: LightGBM Baseline Model (Günlük Re-training & 24 Saat Tahmin)


In [5]:
def train_predict_lgb_baseline(train_df, test_df, features, target):
    model = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.03, verbose=-1, random_state=42)
    model.fit(train_df[features], train_df[target])
    return model.predict(test_df[features])

print("🚀 DENEME 1: LightGBM Baseline 365-Günlük Backtest Başlatılıyor...")
lgb_summary, lgb_daily = run_daily_walk_forward_backtest(df_model, feature_cols, target_col, train_predict_lgb_baseline, max_days=365)

print("=" * 95)
print("📊 DENEME 1: LIGHTGBM BASELINE 3, 6, 9 VE 12 AYLIK GÜNLÜK BACKTEST SONUÇLARI (MAE, MAPE, WAPE)")
print("=" * 95)
print(lgb_summary.to_string(index=False))
print("=" * 95)

🚀 DENEME 1: LightGBM Baseline 365-Günlük Backtest Başlatılıyor...
⏳ [05/365] (  1.4%) | Tarih: 2026-07-24 | MAE: $ 8.87/MWh | WAPE: %12.52
⏳ [10/365] (  2.7%) | Tarih: 2026-07-19 | MAE: $ 9.23/MWh | WAPE: %17.32
⏳ [15/365] (  4.1%) | Tarih: 2026-07-14 | MAE: $ 7.90/MWh | WAPE: %14.59
⏳ [20/365] (  5.5%) | Tarih: 2026-07-09 | MAE: $10.43/MWh | WAPE: %15.08
⏳ [25/365] (  6.8%) | Tarih: 2026-07-04 | MAE: $14.53/MWh | WAPE: %29.57
⏳ [30/365] (  8.2%) | Tarih: 2026-06-29 | MAE: $11.77/MWh | WAPE: %30.20
⏳ [35/365] (  9.6%) | Tarih: 2026-06-24 | MAE: $ 8.38/MWh | WAPE: %18.72
⏳ [40/365] ( 11.0%) | Tarih: 2026-06-19 | MAE: $ 7.79/MWh | WAPE: %51.91
⏳ [45/365] ( 12.3%) | Tarih: 2026-06-14 | MAE: $ 4.71/MWh | WAPE: %41.41
⏳ [50/365] ( 13.7%) | Tarih: 2026-06-09 | MAE: $ 8.28/MWh | WAPE: %25.85
⏳ [55/365] ( 15.1%) | Tarih: 2026-06-04 | MAE: $13.68/MWh | WAPE: %47.62
⏳ [60/365] ( 16.4%) | Tarih: 2026-05-30 | MAE: $ 3.59/MWh | WAPE: %96.09
⏳ [65/365] ( 17.8%) | Tarih: 2026-05-25 | MAE: $11.13/MWh 

## 🧪 DENEME 2 (NİHAİ BAŞARILI): 2026 Rejim Değişimi Uyarlamalı İyileştirilmiş Model

_(1. Hedef Değişken Log-Transform (`log1p(price)` & `expm1`), 2. Baraj Su Enerjisi İmkânı (`hydro_water_energy_mwh`), 3. Güneş/Rüzgar Zirve Oranları Öznitelikleri)_


In [6]:
def train_predict_lgb_log_hydro(train_df, test_df, features, target):
    # Logaritmik hedef değişken eğitimi (Log1p)
    y_tr_log = np.log1p(train_df[target])
    model = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.03, verbose=-1, random_state=42)
    model.fit(train_df[features], y_tr_log)
    
    # Tahminlerin geri dönüştürülmesi (Expm1 & Non-negative Clip)
    preds_log = model.predict(test_df[features])
    preds = np.expm1(preds_log)
    return np.maximum(preds, 0.0)

print("🚀 DENEME 2: İyileştirilmiş Model (Log Transform + Baraj Su İmkânı) 365-Günlük Backtest Başlatılıyor...")
exp2_summary, exp2_daily = run_daily_walk_forward_backtest(df_model, feature_cols, target_col, train_predict_lgb_log_hydro, max_days=365)

print("=" * 95)
print("🏆 DENEME 2: İYİLEŞTİRİLMİŞ MODEL 3, 6, 9 VE 12 AYLIK GÜNLÜK BACKTEST SONUÇLARI (MAE, MAPE, WAPE)")
print("=" * 95)
print(exp2_summary.to_string(index=False))
print("=" * 95)

🚀 DENEME 2: İyileştirilmiş Model (Log Transform + Baraj Su İmkânı) 365-Günlük Backtest Başlatılıyor...
⏳ [05/365] (  1.4%) | Tarih: 2026-07-24 | MAE: $ 9.27/MWh | WAPE: %13.09
⏳ [10/365] (  2.7%) | Tarih: 2026-07-19 | MAE: $11.83/MWh | WAPE: %22.18
⏳ [15/365] (  4.1%) | Tarih: 2026-07-14 | MAE: $ 7.53/MWh | WAPE: %13.91
⏳ [20/365] (  5.5%) | Tarih: 2026-07-09 | MAE: $12.41/MWh | WAPE: %17.94
⏳ [25/365] (  6.8%) | Tarih: 2026-07-04 | MAE: $11.84/MWh | WAPE: %24.10
⏳ [30/365] (  8.2%) | Tarih: 2026-06-29 | MAE: $13.15/MWh | WAPE: %33.73
⏳ [35/365] (  9.6%) | Tarih: 2026-06-24 | MAE: $13.01/MWh | WAPE: %29.05
⏳ [40/365] ( 11.0%) | Tarih: 2026-06-19 | MAE: $ 3.61/MWh | WAPE: %24.04
⏳ [45/365] ( 12.3%) | Tarih: 2026-06-14 | MAE: $ 4.46/MWh | WAPE: %39.24
⏳ [50/365] ( 13.7%) | Tarih: 2026-06-09 | MAE: $11.88/MWh | WAPE: %37.10
⏳ [55/365] ( 15.1%) | Tarih: 2026-06-04 | MAE: $16.41/MWh | WAPE: %57.12
⏳ [60/365] ( 16.4%) | Tarih: 2026-05-30 | MAE: $ 2.45/MWh | WAPE: %65.62
⏳ [65/365] ( 17.8%) |

## 📊 5. Model Denemeleri Karşılaştırma Rehberi

> ℹ️ **Not:** Yukarıdaki kod hücreleri çalıştırıldığında kendi canlı veritabanınız üzerindeki kesin MAE, MAPE ve WAPE sonuçları dinamik olarak basılır.
>
> 💡 **Derin Öğrenme Notu:** PyTorch tabanlı **EPNet (CNN-LSTM)** modelini 365 günlük uzun walk-forward döngüsünde çalıştırmak için terminalden aşağıdaki komutu kullanabilirsiniz:
>
> ```bash
> python run_epnet_backtest.py --days 30
> ```
